## Figure 18a — surface AO from the same vertical NAM product

Plot action: read the canonical processed product(s), apply display-only selection/reshaping, and render this one logical figure as PNG and PDF with one shared stem.


Inputs: the canonical year-0008 vertical NAM and the three canonical
daily verification files. AO is not an independent EOF or calibrated
series: the reference is displayed by selecting 1000 hPa from nam, and
the hindcast means/spreads are read directly from ao_ensemble_mean and
ao_spread, which diagnostic notebook 05 derived from that same 1000-hPa
NAM slice. No February calibration or separate AO path exists.

Outputs: figure18a.png and PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "02_diagnostics").is_dir() and (candidate / "03_plotting").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "runtime"
REPOSITORY_RUNTIME_ROOT = DEFAULT_DERIVED_ROOT.resolve()
PUBLIC_ROOT = Path("/mnt/soclim0/public_data/weiji").resolve()
PROTECTED_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        Path("/mnt/backup_ETH"),
        PUBLIC_ROOT / "B2000WCN001002_timefixed",
        PUBLIC_ROOT / "BWCN",
        PUBLIC_ROOT / "Hindcast",
        PUBLIC_ROOT / "Marina",
        PUBLIC_ROOT / "MERRA2M2I6NPANA",
        PUBLIC_ROOT / "MERRA2_Processed",
        PUBLIC_ROOT / "MLS",
        PUBLIC_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == PUBLIC_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if REPOSITORY_ROOT.is_dir() and not is_within(root, REPOSITORY_RUNTIME_ROOT):
        raise PermissionError(
            "PAPER1_DERIVED_ROOT must remain below this checkout's dedicated "
            f"runtime tree: root={root}, scope={REPOSITORY_RUNTIME_ROOT}"
        )
    for protected in PROTECTED_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored runtime tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run 02_diagnostics on STREAM2 or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
reference = load_dataset(
    "nam/waccm_bwcn_year0008_nam_ao.nc", ("nam", "date")
)
reference_1000 = pressure_slice(reference["nam"], 1000.0)
labels = {
    "0008-01": "January initialization",
    "0008-02": "February initialization",
    "0008-03": "March initialization",
}
colors = {
    "0008-01": "#1f77b4",
    "0008-02": "#2ca02c",
    "0008-03": "#d6278b",
}

def x_axis(date_variable):
    _, month, day = date_parts(date_variable)
    starts = np.array(
        [0, 0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
    )
    return starts[month] + day - 1

figure, axis = plt.subplots(figsize=(13.0, 6.2))
axis.plot(
    x_axis(reference["date"]), reference_1000,
    color="black", lw=2.3, label="WACCM year 0008 reference",
)
for case in ("0008-01", "0008-02", "0008-03"):
    relative = f"verification/{case}_daily.nc"
    product = load_dataset(
        relative, ("ao_ensemble_mean", "ao_spread", "date")
    )
    mean = np.asarray(product["ao_ensemble_mean"].values, dtype=float)
    spread = np.asarray(product["ao_spread"].values, dtype=float)
    x = x_axis(product["date"])
    axis.fill_between(
        x, mean - spread, mean + spread,
        color=colors[case], alpha=0.18,
    )
    axis.plot(
        x, mean, color=colors[case], lw=2.0, label=labels[case]
    )
axis.axhline(0, color="0.55", lw=0.75, ls=":")
axis.set_xlim(0, 150)
axis.set_xticks(
    [0, 31, 59, 90, 120], ["Jan", "Feb", "Mar", "Apr", "May"]
)
axis.set_xlabel("No-leap calendar month")
axis.set_ylabel("AO = vertical NAM at 1000 hPa")
axis.set_title(
    "Reference and hindcast surface AO evolution",
    fontweight="bold",
)
axis.grid(color="0.88", lw=0.5)
axis.legend(ncol=2, frameon=False)
save_figure(figure, "figure18a")
